In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# 5-Class XGBoost Classifier with Combined Feature Engineering (`models/xgboost_all.ipynb`)

This notebook trains a **5-Class Multi-Class XGBoost Classifier** for **ESI 1, 2, 3, 4, and 5** incorporating **29 Predictor Features** (3 baseline features + 10 binary vital anomaly flags + 16 continuous vital delta and range features):

### Predictor Feature Inventory (29 Predictor Features Total)
1. **Baseline Features (3)**: `age`, `gender`, `cc_breathingdifficulty`.
2. **10 Binary Vital Anomaly Flags**:
   - `is_dyspnea_total`: `triage_vital_o2 < 90`
   - `is_dyspnea_moderate`: `triage_vital_o2 >= 90 & triage_vital_o2 < 94`
   - `is_bradypnea`: `triage_vital_rr < 10`
   - `is_tachypnea`: `triage_vital_rr > 30`
   - `is_hypotension`: `triage_vital_sbp <= 90`
   - `is_hypertension`: `triage_vital_sbp > 220`
   - `is_bradycardia_total`: `triage_vital_hr < 40`
   - `is_bradycardia_moderate`: `triage_vital_hr >= 40 & triage_vital_hr < 60`
   - `is_tachycardia_total`: `triage_vital_hr > 150`
   - `is_tachycardia_moderate`: `triage_vital_hr > 100 & triage_vital_hr <= 150`
3. **16 Continuous Vital Delta & Range Features**:
   - `hr_mean_to_last`: `triage_vital_hr - pulse_last`
   - `sbp_mean_to_last`: `triage_vital_sbp - sbp_last`
   - `spo2_mean_to_last`: `triage_vital_o2 - spo2_last`
   - `rr_mean_to_last`: `triage_vital_rr - resp_last`
   - `hr_range`: `pulse_max - pulse_min`
   - `rr_range`: `resp_max - resp_min`
   - `spo2_range`: `spo2_max - spo2_min`
   - `sbp_range`: `sbp_max - sbp_min`
   - `hr_last_to_min`: `pulse_last - pulse_min`
   - `rr_last_to_min`: `resp_last - resp_min`
   - `spo2_last_to_min`: `spo2_last - spo2_min`
   - `sbp_last_to_min`: `sbp_last - sbp_min`
   - `hr_last_to_max`: `pulse_last - pulse_max`
   - `rr_last_to_max`: `resp_last - resp_max`
   - `spo2_last_to_max`: `spo2_last - spo2_max`
   - `sbp_last_to_max`: `sbp_last - sbp_max`

### Evaluated Metrics
- **Accuracy** (Overall 5-class accuracy)
- **Precision** (Macro Precision & per-class precision)
- **Recall** (Macro Recall / Sensitivity & per-class recall)
- **ROC-AUC** (Macro One-vs-Rest ROC-AUC & per-class ROC-AUC)
- **F1-Score** (Macro F1 & per-class F1-Score)
- **Confusion Matrix** (5x5 confusion matrix)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(caret)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pROC)
  library(xgboost)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}
config <- fromJSON(config_path)
cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data & Construct 29 Predictor Features
# ---------------------------------------------------------
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}
cat("Loading dataset from:", data_file, "...\n")
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))
raw_df <- get(data_obj_name, envir = data_env)
target_col <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
# Helper vectors for vital trends and ranges
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
p_last   <- get_vec("pulse_last")
p_max    <- get_vec("pulse_max")
p_min    <- get_vec("pulse_min")
s_last   <- get_vec("sbp_last")
s_max    <- get_vec("sbp_max")
s_min    <- get_vec("sbp_min")
o2_last  <- get_vec("spo2_last")
o2_max   <- get_vec("spo2_max")
o2_min   <- get_vec("spo2_min")
r_last   <- get_vec("resp_last")
r_max    <- get_vec("resp_max")
r_min    <- get_vec("resp_min")
t_hr     <- get_vec("triage_vital_hr")
t_sbp    <- get_vec("triage_vital_sbp")
t_o2     <- get_vec("triage_vital_o2")
t_rr     <- get_vec("triage_vital_rr")
# Construct 29 Features: 3 baseline + 10 binary anomaly flags + 16 vital delta & range features
df_full <- data.frame(
  # 1. Baseline Features (3)
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  
  # 2. Binary Vital Anomaly Flags (10)
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0),
  
  # 3. Continuous Vital Delta & Range Features (16)
  hr_mean_to_last         = t_hr - p_last,
  sbp_mean_to_last        = t_sbp - s_last,
  spo2_mean_to_last       = t_o2 - o2_last,
  rr_mean_to_last         = t_rr - r_last,
  
  hr_range                = p_max - p_min,
  rr_range                = r_max - r_min,
  spo2_range              = o2_max - o2_min,
  sbp_range               = s_max - s_min,
  
  hr_last_to_min          = p_last - p_min,
  rr_last_to_min          = r_last - r_min,
  spo2_last_to_min        = o2_last - o2_min,
  sbp_last_to_min         = s_last - s_min,
  
  hr_last_to_max          = p_last - p_max,
  rr_last_to_max          = r_last - r_max,
  spo2_last_to_max        = o2_last - o2_max,
  sbp_last_to_max         = s_last - s_max
)
raw_esi <- as.character(raw_df[[target_col]])
df_full$target_col <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
initial_rows <- nrow(df_full)
df_full <- na.omit(df_full)
cat(sprintf("Complete Case Filtering: Removed %d rows with NULL/NA features (Remaining complete rows: %d)\n",
            initial_rows - nrow(df_full), nrow(df_full)))
cat(sprintf("Full Feature Dataset Ready: %d total rows x %d cols\n", nrow(df_full), ncol(df_full)))
cat("Predictor Features Included (29 Total Features):\n")
print(setdiff(names(df_full), "target_col"))
cat("\nNatural 5-Class Target Distribution ('1', '2', '3', '4', '5'):\n")
print(table(df_full$target_col))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Data Partitioning & Scaling
# ---------------------------------------------------------
set.seed(config$training$random_state)
test_size <- config$training$test_size
val_size  <- config$training$val_size
# Stratified Test split (15%)
in_train_val <- createDataPartition(df_full$target_col, p = 1 - test_size, list = FALSE)
train_val_df <- df_full[in_train_val, ]
test_df      <- df_full[-in_train_val, ]
# Stratified Validation split (15%)
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_col, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]
# Standardize continuous features across splits
binary_cols <- c("gender", "cc_breathingdifficulty",
                 "is_dyspnea_total", "is_dyspnea_moderate", "is_bradypnea", "is_tachypnea",
                 "is_hypotension", "is_hypertension", "is_bradycardia_total", "is_bradycardia_moderate",
                 "is_tachycardia_total", "is_tachycardia_moderate")
cont_cols <- setdiff(names(train_df), c(binary_cols, "target_col"))
preproc   <- preProcess(train_df[, cont_cols, drop = FALSE], method = c("center", "scale"))
train_df <- predict(preproc, train_df)
val_df   <- predict(preproc, val_df)
test_df  <- predict(preproc, test_df)
feat_names <- setdiff(names(train_df), "target_col")
X_train <- as.matrix(train_df[, feat_names])
y_train_0idx <- as.integer(train_df$target_col) - 1
X_val   <- as.matrix(val_df[, feat_names])
y_val_0idx   <- as.integer(val_df$target_col) - 1
X_test  <- as.matrix(test_df[, feat_names])
y_test_0idx  <- as.integer(test_df$target_col) - 1
dtrain_xgb <- xgb.DMatrix(data = X_train, label = y_train_0idx)
dval_xgb   <- xgb.DMatrix(data = X_val, label = y_val_0idx)
dtest_xgb  <- xgb.DMatrix(data = X_test, label = y_test_0idx)
cat("=== Target Distributions Across Splits ===\n")
cat("Training Set Target Distribution:\n")
print(table(train_df$target_col))
cat("\nValidation Set Target Distribution:\n")
print(table(val_df$target_col))
cat("\nTest Set Target Distribution:\n")
print(table(test_df$target_col))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Train 5-Class Multi-Class XGBoost Model
# ---------------------------------------------------------
set.seed(config$training$random_state)
cat("Training 5-Class Multi-Class XGBoost Model (ESI 1, 2, 3, 4, 5)...\n")
xgb_params <- list(
  objective        = "multi:softprob",
  num_class        = 5,
  eval_metric      = "mlogloss",
  eta              = 0.05,
  max_depth        = 6,
  subsample        = 0.8,
  colsample_bytree = 0.8
)
xgb_model <- xgb.train(
  params                = xgb_params,
  data                  = dtrain_xgb,
  nrounds               = 150,
  evals                 = list(train = dtrain_xgb, val = dval_xgb),
  early_stopping_rounds = 20,
  verbose               = 0
)
cat("Multi-Class XGBoost Training Complete!\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Comprehensive Benchmark Across Splits & Report Metrics (Accuracy, Precision, Recall, ROC-AUC, F1-Score, Confusion Matrix)
# ---------------------------------------------------------
evaluate_xgboost_5class <- function(model, dmatrix, actual_factor, set_name) {
  raw_preds <- predict(model, newdata = dmatrix)
  prob_mat  <- matrix(raw_preds, ncol = 5, byrow = TRUE)
  colnames(prob_mat) <- c("1", "2", "3", "4", "5")
  
  pred_indices <- apply(prob_mat, 1, which.max)
  pred_val     <- colnames(prob_mat)[pred_indices]
  pred_fac     <- factor(pred_val, levels = c("1", "2", "3", "4", "5"))
  act_fac      <- factor(actual_factor, levels = c("1", "2", "3", "4", "5"))
  
  cm  <- confusionMatrix(pred_fac, act_fac)
  acc <- as.numeric(cm$overall["Accuracy"])
  
  prec_by_class <- as.numeric(cm$byClass[, "Pos Pred Value"])
  rec_by_class  <- as.numeric(cm$byClass[, "Sensitivity"])
  prec_by_class[is.na(prec_by_class)] <- 0
  rec_by_class[is.na(rec_by_class)]   <- 0
  
  f1_by_class <- ifelse((prec_by_class + rec_by_class) > 0, 
                        2 * (prec_by_class * rec_by_class) / (prec_by_class + rec_by_class), 0)
  
  roc_auc_by_class <- sapply(1:5, function(i) {
    cls_name <- levels(act_fac)[i]
    act_bin  <- ifelse(act_fac == cls_name, 1, 0)
    r_obj    <- tryCatch(pROC::roc(act_bin, prob_mat[, i]), error = function(e) NULL)
    if (!is.null(r_obj)) as.numeric(r_obj$auc) else NA
  })
  
  actual_counts <- as.numeric(table(act_fac))
  pred_counts   <- as.numeric(table(pred_fac))
  diff_vec      <- pred_counts - actual_counts
  diff_str      <- ifelse(diff_vec >= 0, paste0("+", diff_vec), as.character(diff_vec))
  
  report_df <- data.frame(
    Class        = levels(act_fac),
    Actual_Count = actual_counts,
    Pred_Count   = pred_counts,
    Diff         = diff_str,
    Precision    = round(prec_by_class, 4),
    Recall       = round(rec_by_class, 4),
    F1_Score     = round(f1_by_class, 4),
    ROC_AUC      = round(roc_auc_by_class, 4)
  )
  
  macro_prec    <- mean(prec_by_class)
  macro_rec     <- mean(rec_by_class)
  macro_f1      <- mean(f1_by_class)
  macro_roc_auc <- mean(roc_auc_by_class, na.rm = TRUE)
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   5-CLASS XGBOOST (FEATURE ENGINEERED) - %s SET BENCHMARK\n", toupper(set_name)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Overall Accuracy     : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  Macro Precision      : %.4f\n", macro_prec))
  cat(sprintf("  Macro Recall (Sens)  : %.4f\n", macro_rec))
  cat(sprintf("  Macro F1-Score       : %.4f\n", macro_f1))
  cat(sprintf("  Macro ROC-AUC        : %.4f\n", macro_roc_auc))
  cat(sprintf("============================================================\n\n"))
  
  cat("Per-Class Performance & Count Summary:\n")
  print(report_df)
  cat("\nConfusion Matrix (Rows: Predicted, Columns: Actual):\n")
  print(cm$table)
  cat(sprintf("============================================================\n\n"))
  
  return(list(acc = acc, macro_prec = macro_prec, macro_rec = macro_rec, macro_f1 = macro_f1, macro_roc_auc = macro_roc_auc, report_df = report_df))
}
res_train <- evaluate_xgboost_5class(xgb_model, dtrain_xgb, train_df$target_col, "Train")
res_val   <- evaluate_xgboost_5class(xgb_model, dval_xgb,   val_df$target_col,   "Validation")
res_test  <- evaluate_xgboost_5class(xgb_model, dtest_xgb,  test_df$target_col,  "Test")
# Write CSV Reports
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
write.csv(res_val$report_df,  file = file.path(reports_dir, "xgboost_all_val_report.csv"),  row.names = FALSE)
write.csv(res_test$report_df, file = file.path(reports_dir, "xgboost_all_test_report.csv"), row.names = FALSE)
cat("Validation CSV Report written to: reports/xgboost_all_val_report.csv\n")
cat("Test CSV Report written to:       reports/xgboost_all_test_report.csv\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Diagnostic Plots (Metrics Bar Chart)
# ---------------------------------------------------------
plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)
metrics_summary <- data.frame(
  Split           = factor(c("Train", "Validation", "Test"), levels = c("Train", "Validation", "Test")),
  Accuracy        = c(res_train$acc,           res_val$acc,           res_test$acc),
  Macro_Precision = c(res_train$macro_prec,     res_val$macro_prec,     res_test$macro_prec),
  Macro_Recall    = c(res_train$macro_rec,      res_val$macro_rec,      res_test$macro_rec),
  Macro_F1_Score  = c(res_train$macro_f1,       res_val$macro_f1,       res_test$macro_f1),
  Macro_ROC_AUC   = c(res_train$macro_roc_auc,  res_val$macro_roc_auc,  res_test$macro_roc_auc)
)
metrics_long <- metrics_summary %>%
  pivot_longer(cols = c("Accuracy", "Macro_Precision", "Macro_Recall", "Macro_F1_Score", "Macro_ROC_AUC"), names_to = "Metric", values_to = "Score")
p_bar <- ggplot(metrics_long, aes(x = Metric, y = Score, fill = Split)) +
  geom_bar(stat = "identity", position = position_dodge(width = 0.7), width = 0.6) +
  geom_text(aes(label = sprintf("%.3f", Score)), position = position_dodge(width = 0.7), vjust = -0.3, size = 3) +
  theme_minimal() +
  scale_fill_manual(values = c("Train" = "#2b5c8f", "Validation" = "#e07a5f", "Test" = "#81b29a")) +
  labs(title = "Train vs. Validation vs. Test Metrics (Feature Engineered XGBoost)",
       subtitle = "29 Predictor Features (Accuracy, Precision, Recall, F1-Score, ROC-AUC)",
       y = "Metric Value Score", x = "") +
  theme(plot.title = element_text(face = "bold", size = 13), legend.position = "top")
ggsave(file.path(plots_dir, "xgboost_all_metrics_barchart.png"), plot = p_bar, width = 10, height = 5, dpi = 300)
cat("Metrics Comparison Bar Chart saved to: plots/xgboost_all_metrics_barchart.png\n")
p_bar

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7: Save Multi-Class XGBoost Model Artifact
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)
model_path <- file.path(deploy_dir, "xgboost_all_model.rds")
saveRDS(list(model = xgb_model, preproc = preproc), file = model_path)
cat("Multi-Class XGBoost Model saved to:", model_path, "\n")